# Amazon Electronics — Dataset Inspection
Quick structural scan of both files before running UMAG.
**Run this first, paste the output back to Claude.**

In [ ]:
import ast, json, os
from collections import Counter, defaultdict
from pathlib import Path


# Add project root to path so src/ is importable
sys.path.insert(0, str(Path('..').resolve()))
from src.utils.paths import P

ROOT = Path("/Volumes/T5 EVO/hf/amazon_electronics")
REVIEWS = ROOT / "reviews_Electronics_5.json"
META    = ROOT / "meta_Electronics.json"

def parse_line(line):
    line = line.strip()
    if not line:
        return None
    try:
        return json.loads(line)
    except json.JSONDecodeError:
        return ast.literal_eval(line)

print("Files exist:", REVIEWS.exists(), META.exists())
print(f"Reviews size: {REVIEWS.stat().st_size / 1e6:.1f} MB")
print(f"Meta    size: {META.stat().st_size    / 1e6:.1f} MB")

Files exist: True True
Reviews size: 1479.0 MB
Meta    size: 663.6 MB


## 1 — Reviews file: keys, sample records, field types

In [2]:
# ── Print first 3 raw lines ──
print("=" * 60)
print("RAW LINES (reviews) — first 3")
print("=" * 60)
with open(REVIEWS, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        rec = parse_line(line)
        print(json.dumps(rec, indent=2, default=str))
        print()

RAW LINES (reviews) — first 3
{
  "reviewerID": "AO94DHGC771SJ",
  "asin": "0528881469",
  "reviewerName": "amazdnu",
  "helpful": [
    0,
    0
  ],
  "reviewText": "We got this GPS for my husband who is an (OTR) over the road trucker.  Very Impressed with the shipping time, it arrived a few days earlier than expected...  within a week of use however it started freezing up... could of just been a glitch in that unit.  Worked great when it worked!  Will work great for the normal person as well but does have the \"trucker\" option. (the big truck routes - tells you when a scale is coming up ect...)  Love the bigger screen, the ease of use, the ease of putting addresses into memory.  Nothing really bad to say about the unit with the exception of it freezing which is probably one in a million and that's just my luck.  I contacted the seller and within minutes of my email I received a email back with instructions for an exchange! VERY impressed all the way around!",
  "overall": 5.0,
  "s

In [ ]:
# ── Scan ALL reviews: collect keys, types, value ranges ──
print("Scanning all reviews (streaming) ...")

key_counter   = Counter()          # how many records have each key
key_types     = defaultdict(set)   # set of Python types per key
overall_vals  = Counter()          # distribution of 'overall' rating
users         = set()
items         = set()
timestamps    = []
total         = 0
null_counts   = defaultdict(int)

with open(REVIEWS, "r", encoding="utf-8") as f:
    for line in f:
        rec = parse_line(line)
        if rec is None:
            continue
        total += 1

        for k, v in rec.items():
            key_counter[k] += 1
            key_types[k].add(type(v).__name__)
            if v is None or v == "":
                null_counts[k] += 1

        if "overall" in rec:
            overall_vals[rec["overall"]] += 1
        if "reviewerID" in rec and rec["reviewerID"]:
            users.add(rec["reviewerID"])
        if "asin" in rec and rec["asin"]:
            items.add(rec["asin"])
        if "unixReviewTime" in rec and rec["unixReviewTime"]:
            timestamps.append(int(rec["unixReviewTime"]))

print(f"\nTotal records : {total:,}")
print(f"Unique users  : {len(users):,}")
print(f"Unique items  : {len(items):,}")

print("\n── Keys (coverage out of", total, "records) ──")
for k, cnt in sorted(key_counter.items(), key=lambda x: -x[1]):
    types_str = ", ".join(sorted(key_types[k]))
    null_str  = f"  nulls={null_counts[k]:,}" if null_counts[k] > 0 else ""
    print(f"  {k:<25} {cnt:>9,} / {total:,}   types=[{types_str}]{null_str}")

In [ ]:
# ── Rating distribution ──
print("\n── 'overall' rating distribution ──")
for rating in sorted(overall_vals):
    cnt  = overall_vals[rating]
    pct  = 100 * cnt / total
    bar  = "█" * int(pct / 1)
    print(f"  {rating:.1f}★  {cnt:>7,}  ({pct:5.1f}%)  {bar}")

pos = sum(v for k, v in overall_vals.items() if k >= 4.0)
neg = sum(v for k, v in overall_vals.items() if k < 4.0)
print(f"\n  Positive (≥4★): {pos:,} ({100*pos/total:.1f}%)")
print(f"  Negative (<4★): {neg:,} ({100*neg/total:.1f}%)")

In [ ]:
# ── Timestamp range ──
import datetime
if timestamps:
    ts_min = min(timestamps)
    ts_max = max(timestamps)
    print(f"\n── Timestamp range ──")
    print(f"  Min: {ts_min}  →  {datetime.datetime.utcfromtimestamp(ts_min).date()}")
    print(f"  Max: {ts_max}  →  {datetime.datetime.utcfromtimestamp(ts_max).date()}")
    span = (ts_max - ts_min) / (365.25 * 24 * 3600)
    print(f"  Span: {span:.1f} years")

In [ ]:
# ── User activity distribution ──
print("\nScanning user activity distribution ...")
user_counts = Counter()
with open(REVIEWS, "r", encoding="utf-8") as f:
    for line in f:
        rec = parse_line(line)
        if rec and rec.get("reviewerID"):
            user_counts[rec["reviewerID"]] += 1

counts = list(user_counts.values())
import statistics
print(f"  Min reviews/user : {min(counts)}")
print(f"  Max reviews/user : {max(counts)}")
print(f"  Mean             : {statistics.mean(counts):.1f}")
print(f"  Median           : {statistics.median(counts):.1f}")

for threshold in [5, 10, 20, 50]:
    pct = 100 * sum(1 for c in counts if c <= threshold) / len(counts)
    print(f"  Users with ≤{threshold:2d} reviews: {pct:.1f}%")

## 2 — Meta file: keys and sample

In [4]:
# ── First 2 meta records ──
print("=" * 60)
print("RAW LINES (meta) — first 2")
print("=" * 60)
with open(META, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 2:
            break
        rec = parse_line(line)
        print(json.dumps(rec, indent=2, default=str))
        print()

# ── Meta keys ──
print("=" * 60)
print("META KEY COVERAGE (first 10,000 records)")
print("=" * 60)
meta_keys = Counter()
meta_types = defaultdict(set)
n = 0
with open(META, "r", encoding="utf-8") as f:
    for line in f:
        if n >= 10_000:
            break
        rec = parse_line(line)
        if rec is None:
            continue
        n += 1
        for k, v in rec.items():
            meta_keys[k] += 1
            meta_types[k].add(type(v).__name__)

for k, cnt in sorted(meta_keys.items(), key=lambda x: -x[1]):
    types_str = ", ".join(sorted(meta_types[k]))
    print(f"  {k:<25} {cnt:>7,} / {n:,}   [{types_str}]")

RAW LINES (meta) — first 2
{
  "asin": "0132793040",
  "imUrl": "http://ecx.images-amazon.com/images/I/31JIPhp%2BGIL.jpg",
  "description": "The Kelby Training DVD Mastering Blend Modes in Adobe Photoshop CS5 with Corey Barker is a useful tool for becoming familiar with the use of blend modes in Adobe Photoshop. For those who are serious about mastering all that Photoshop has to offer, mastering blend modes is just as important as mastering layers.In this DVD tutorial, seasoned expert Corey Barker explores the function of blend modes in a variety of scenarios such as image restoration, sharpening, adjustments, special effects and much more. Since every project scenario is different, Corey encourages you to experiment with these blend modes by giving you the skills and confidence you need.",
  "categories": [
    [
      "Electronics",
      "Computers & Accessories",
      "Cables & Accessories",
      "Monitor Accessories"
    ]
  ],
  "title": "Kelby Training DVD: Mastering Blend Mod